In [70]:
from dotenv import load_dotenv
from pathlib import Path
import sys
import os

# Walk up until we find the project root (folder with the .env)
current_path = Path().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        project_root = os.getenv("PROJECT_ROOT")
        print(project_root)
        sys.path.append(project_root)     
        break


%load_ext autoreload
%autoreload 2

/mnt/home/test/beluga-call-pipeline/
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [71]:
import pandas as pd
labels_v1 = pd.read_csv("../../data/Verified_Dataset/labels/labels_eval_v1.csv")
labels_v2 = pd.read_csv("../../data/Verified_Dataset/labels/labels_eval_v2.csv")
irene_20min = pd.read_csv("../../data/Verified_Dataset/labels/labels_irene_20min.csv")
irene_5min = pd.read_csv("../../data/Verified_Dataset/labels/labels_irene_5min.csv")
old_abs_df = pd.read_csv("../../data/Verified_Dataset/labels/labels_old_abs.csv")

In [72]:
labels_v1["labeling_effort"] = "evaluation_v1"
# labels_v2["labeling_effort"] = "evaluation_v2" 
labels_v2["labeling_effort"] = labels_v2.apply(
    lambda row: "abs_v2_emmanuel" if str(row.get("annotator", "")) == "EM" else "evaluation_v2",
    axis=1
) 
irene_20min["labeling_effort"] = "irene_20min"
irene_5min["labeling_effort"] = "irene_5min"

In [73]:
old_abs_df["labeling_effort"] = "old_abs_from_3s_snippets"
old_abs_df.loc[old_abs_df["Origin"] == "1s_absences", "labeling_effort"] = "old_abs_CAC_last_samples"
old_abs_df.drop(columns=["Origin"], inplace=True)

In [74]:
labels_v1["BBPC"].value_counts()

BBPC
0    3393
1     208
Name: count, dtype: int64

In [75]:
# Merge the two dataframes, keeping all columns, filling missing values with empty where necessary
merged_df = pd.concat([labels_v1, labels_v2, irene_20min, irene_5min, old_abs_df], ignore_index=True, sort=False)
# merged_df = pd.concat([labels_v1, irene_20min, old_abs_df], ignore_index=True, sort=False)
# Fill all NaN values with empty string for reporting/compatibility
merged_df = merged_df.fillna('')
# Show info and preview to check result
print("Merged DataFrame shape:", merged_df.shape)
display(merged_df.head())

Merged DataFrame shape: (17389, 30)


,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,clip_filename,ECHO,BBPC,HFPC,Whistle,Boat,labeling_effort,DETAILS,boat_labeling_file,boat_labeling_file_id,annotator,DETAIL,SnippetFilename,start_s,end_s
0,0.0,1.0,a,,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536.0,1136.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:38:58.000000000,2017-07-24 13:38:59.000000000,BSM_20170724_13385800.wav,0,0,0,0,1.0,evaluation_v1,,,,,,,,
1,1.0,2.0,e,,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536.0,1136.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:38:59.000000000,2017-07-24 13:39:00.000000000,BSM_20170724_13385900.wav,1,0,0,0,1.0,evaluation_v1,,,,,,,,
2,2.0,3.0,e,,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536.0,1136.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:39:00.000000000,2017-07-24 13:39:01.000000000,BSM_20170724_13390000.wav,1,0,0,0,1.0,evaluation_v1,,,,,,,,
3,3.0,4.0,e,,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536.0,1136.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:39:01.000000000,2017-07-24 13:39:02.000000000,BSM_20170724_13390100.wav,1,0,0,0,1.0,evaluation_v1,,,,,,,,
4,4.0,5.0,e,,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536.0,1136.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:39:02.000000000,2017-07-24 13:39:03.000000000,BSM_20170724_13390200.wav,1,0,0,0,1.0,evaluation_v1,,,,,,,,


In [76]:
merged_df["Call"] = merged_df[["ECHO", "BBPC", "HFPC", "Whistle"]].any(axis=1).astype(int)


In [77]:
pd.set_option('display.max_columns', None)

In [78]:
# Define the desired order for the first 7 columns
first_columns = ['clip_filename', 'ECHO', 'BBPC', 'HFPC', 'Whistle', 'Call', 'Boat', 'labeling_effort', 'DETAIL']
# Get the remaining columns not in the first_columns list
remaining_columns = [col for col in merged_df.columns if col not in first_columns]
# Reorder the dataframe
merged_df = merged_df[first_columns + remaining_columns]
merged_df

,clip_filename,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,DETAIL,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,DETAILS,boat_labeling_file,boat_labeling_file_id,annotator,SnippetFilename,start_s,end_s
0,BSM_20170724_13385800.wav,0,0,0,0,0,1.0,evaluation_v1,,0.0,1.0,a,,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536.0,1136.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:38:58.000000000,2017-07-24 13:38:59.000000000,,,,,,,
1,BSM_20170724_13385900.wav,1,0,0,0,1,1.0,evaluation_v1,,1.0,2.0,e,,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536.0,1136.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:38:59.000000000,2017-07-24 13:39:00.000000000,,,,,,,
2,BSM_20170724_13390000.wav,1,0,0,0,1,1.0,evaluation_v1,,2.0,3.0,e,,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536.0,1136.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:39:00.000000000,2017-07-24 13:39:01.000000000,,,,,,,
3,BSM_20170724_13390100.wav,1,0,0,0,1,1.0,evaluation_v1,,3.0,4.0,e,,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536.0,1136.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:39:01.000000000,2017-07-24 13:39:02.000000000,,,,,,,
4,BSM_20170724_13390200.wav,1,0,0,0,1,1.0,evaluation_v1,,4.0,5.0,e,,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536.0,1136.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:39:02.000000000,2017-07-24 13:39:03.000000000,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17384,CAC_20210714_09242300.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 09:24:23.000000,,,CAC,,201359382,-172.7,2021-07-14 09:24:23.000000,2021-07-14 09:24:24.000000,,,,,CAC_20210714_09242300.wav,-1.0,0.0
17385,CAC_20210714_09242800.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 09:24:28.000000,,,CAC,,201359382,-172.7,2021-07-14 09:24:28.000000,2021-07-14 09:24:29.000000,,,,,CAC_20210714_09242800.wav,-1.0,0.0
17386,CAC_20210714_09251600.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 09:25:16.000000,,,CAC,,201359382,-172.7,2021-07-14 09:25:16.000000,2021-07-14 09:25:17.000000,,,,,CAC_20210714_09251600.wav,-1.0,0.0
17387,CAC_20210714_09271800.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 09:27:18.000000,,,CAC,,201359382,-172.7,2021-07-14 09:27:18.000000,2021-07-14 09:27:19.000000,,,,,CAC_20210714_09271800.wav,-1.0,0.0


In [79]:
print(merged_df["ECHO"].value_counts(dropna=False))
print(merged_df["BBPC"].value_counts(dropna=False))
print(merged_df["HFPC"].value_counts(dropna=False))
print(merged_df["Whistle"].value_counts(dropna=False))
print(merged_df["Call"].value_counts(dropna=False))
print(merged_df["Boat"].value_counts(dropna=False))


ECHO
0    12005
1     5384
Name: count, dtype: int64
BBPC
0    16401
1      988
Name: count, dtype: int64
HFPC
0    16594
1      795
Name: count, dtype: int64
Whistle
0    14464
1     2925
Name: count, dtype: int64
Call
0    10628
1     6761
Name: count, dtype: int64
Boat
       6869
1.0    6158
0.0    4362
Name: count, dtype: int64


In [80]:
merged_df

,clip_filename,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,DETAIL,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,DETAILS,boat_labeling_file,boat_labeling_file_id,annotator,SnippetFilename,start_s,end_s
0,BSM_20170724_13385800.wav,0,0,0,0,0,1.0,evaluation_v1,,0.0,1.0,a,,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536.0,1136.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:38:58.000000000,2017-07-24 13:38:59.000000000,,,,,,,
1,BSM_20170724_13385900.wav,1,0,0,0,1,1.0,evaluation_v1,,1.0,2.0,e,,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536.0,1136.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:38:59.000000000,2017-07-24 13:39:00.000000000,,,,,,,
2,BSM_20170724_13390000.wav,1,0,0,0,1,1.0,evaluation_v1,,2.0,3.0,e,,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536.0,1136.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:39:00.000000000,2017-07-24 13:39:01.000000000,,,,,,,
3,BSM_20170724_13390100.wav,1,0,0,0,1,1.0,evaluation_v1,,3.0,4.0,e,,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536.0,1136.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:39:01.000000000,2017-07-24 13:39:02.000000000,,,,,,,
4,BSM_20170724_13390200.wav,1,0,0,0,1,1.0,evaluation_v1,,4.0,5.0,e,,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536.0,1136.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:39:02.000000000,2017-07-24 13:39:03.000000000,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17384,CAC_20210714_09242300.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 09:24:23.000000,,,CAC,,201359382,-172.7,2021-07-14 09:24:23.000000,2021-07-14 09:24:24.000000,,,,,CAC_20210714_09242300.wav,-1.0,0.0
17385,CAC_20210714_09242800.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 09:24:28.000000,,,CAC,,201359382,-172.7,2021-07-14 09:24:28.000000,2021-07-14 09:24:29.000000,,,,,CAC_20210714_09242800.wav,-1.0,0.0
17386,CAC_20210714_09251600.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 09:25:16.000000,,,CAC,,201359382,-172.7,2021-07-14 09:25:16.000000,2021-07-14 09:25:17.000000,,,,,CAC_20210714_09251600.wav,-1.0,0.0
17387,CAC_20210714_09271800.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 09:27:18.000000,,,CAC,,201359382,-172.7,2021-07-14 09:27:18.000000,2021-07-14 09:27:19.000000,,,,,CAC_20210714_09271800.wav,-1.0,0.0


# Verifying all wavs are generated and 1 second long

In [81]:
# Only consider duplicates for rows where labeling_effort is "irene_20min" or "evaluation_v1"
# effort_mask = merged_df["labeling_effort"].isin(["irene_20min", "evaluation_v1"])

efforts = merged_df["labeling_effort"].unique()
effort_mask = merged_df["labeling_effort"].isin(efforts)
effort_df = merged_df[effort_mask]

dupes_restricted = effort_df[effort_df['clip_filename'].duplicated(keep=False)].copy()

# Columns to check for label consistency
cols_to_check_restricted = ['ECHO', 'BBPC', 'HFPC', 'Whistle']

def has_mismatched_labels_restricted(group):
    # True if at least two rows in the group have different label values across the columns of interest
    return group[cols_to_check_restricted].nunique(dropna=False).max() > 1

# Compute mismatch flag for each group using apply(), then map back to the DataFrame
mismatch_by_clip = dupes_restricted.groupby('clip_filename').apply(has_mismatched_labels_restricted)
dupes_restricted["label_mismatch"] = dupes_restricted['clip_filename'].map(mismatch_by_clip)

print(dupes_restricted["label_mismatch"].value_counts())
sorted_dupes = dupes_restricted.sort_values(by='clip_filename')
display(sorted_dupes[sorted_dupes["clip_filename"] == "CAC_20210714_08384800.wav"].head(20))

diff_groups_restricted = dupes_restricted[dupes_restricted["label_mismatch"] == True]

if not diff_groups_restricted.empty:
    print("Duplicated clip_filenames (restricted to irene_20min or evaluation_v1) with differing ECHO, BBPC, HFPC, or Whistle values:")
    diff_groups_restricted = diff_groups_restricted.sort_values(by='clip_filename')
    display(diff_groups_restricted.head(10))
else:
    print("All duplicated clip_filenames (irene_20min or evaluation_v1) have identical ECHO, BBPC, HFPC, and Whistle values.")

label_mismatch
False    1070
True       36
Name: count, dtype: int64


,clip_filename,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,DETAIL,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,DETAILS,boat_labeling_file,boat_labeling_file_id,annotator,SnippetFilename,start_s,end_s,label_mismatch
9530,CAC_20210714_08384800.wav,0,0,1,1,1,1.0,irene_20min,hm,,,hw,ship,,,2021-07-14 08:38:47.007000,,,CAC,,201359382,-172.7,2021-07-14 08:38:48.007000,2021-07-14 08:38:49.007000,,,,,CAC_20210714_08384700.wav,1.0,2.0,True
2275,CAC_20210714_08384800.wav,1,1,1,1,1,1.0,evaluation_v1,,475.0,476.0,ewbh,,201359382.210714075958.wav,201359382.210714083053.snippet.wav,2021-07-14 08:30:53,1855.0,2455.0,CAC,../../data/evaluation_snippets/v1/CAC_2021,201359382,-172.7,2021-07-14 08:38:48.000000000,2021-07-14 08:38:49.000000000,,,,,,,,True


Duplicated clip_filenames (restricted to irene_20min or evaluation_v1) with differing ECHO, BBPC, HFPC, or Whistle values:


,clip_filename,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,DETAIL,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,DETAILS,boat_labeling_file,boat_labeling_file_id,annotator,SnippetFilename,start_s,end_s,label_mismatch
9126,BSM_20170724_13562209.wav,1,0,0,0,1,1.0,irene_20min,,,,e,ship,,,2017-07-24 13:56:20.090000,,,BSM,,201359382,-172.7,2017-07-24 13:56:22.090000,2017-07-24 13:56:23.090000,,,,,BSM_20170724_13562009.wav,2.0,3.0,True
10526,BSM_20170724_13562209.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,,,2017-07-24 13:56:20.090,,,BSM,,201359382,-172.7,2017-07-24 13:56:22.090,2017-07-24 13:56:23.090,,,,,BSM_20170724_13562009.wav,1.0,2.0,True
10536,BSM_20170801_19043620.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,,,2017-08-01 19:04:35.200,,,BSM,,201359382,-172.7,2017-08-01 19:04:36.200,2017-08-01 19:04:37.200,,,,,BSM_20170801_19043520.wav,0.0,1.0,True
9206,BSM_20170801_19043620.wav,1,0,0,0,1,0.0,irene_20min,,,,e,,,,2017-08-01 19:04:35.200000,,,BSM,,201359382,-172.7,2017-08-01 19:04:36.200000,2017-08-01 19:04:37.200000,,,,,BSM_20170801_19043520.wav,1.0,2.0,True
17220,CAC_20210714_08305700.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 08:30:57.000000,,,CAC,,201359382,-172.7,2021-07-14 08:30:57.000000,2021-07-14 08:30:58.000000,,,,,CAC_20210714_08305700.wav,-1.0,0.0,True
1804,CAC_20210714_08305700.wav,1,0,0,0,1,1.0,evaluation_v1,,4.0,5.0,e,,201359382.210714075958.wav,201359382.210714083053.snippet.wav,2021-07-14 08:30:53,1855.0,2455.0,CAC,../../data/evaluation_snippets/v1/CAC_2021,201359382,-172.7,2021-07-14 08:30:57.000000000,2021-07-14 08:30:58.000000000,,,,,,,,True
17227,CAC_20210714_08314800.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 08:31:48.000000,,,CAC,,201359382,-172.7,2021-07-14 08:31:48.000000,2021-07-14 08:31:49.000000,,,,,CAC_20210714_08314800.wav,-1.0,0.0,True
1855,CAC_20210714_08314800.wav,1,0,0,0,1,1.0,evaluation_v1,,55.0,56.0,e,,201359382.210714075958.wav,201359382.210714083053.snippet.wav,2021-07-14 08:30:53,1855.0,2455.0,CAC,../../data/evaluation_snippets/v1/CAC_2021,201359382,-172.7,2021-07-14 08:31:48.000000000,2021-07-14 08:31:49.000000000,,,,,,,,True
17228,CAC_20210714_08320100.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 08:32:01.000000,,,CAC,,201359382,-172.7,2021-07-14 08:32:01.000000,2021-07-14 08:32:02.000000,,,,,CAC_20210714_08320100.wav,-1.0,0.0,True
1868,CAC_20210714_08320100.wav,1,0,0,0,1,1.0,evaluation_v1,,68.0,69.0,e,,201359382.210714075958.wav,201359382.210714083053.snippet.wav,2021-07-14 08:30:53,1855.0,2455.0,CAC,../../data/evaluation_snippets/v1/CAC_2021,201359382,-172.7,2021-07-14 08:32:01.000000000,2021-07-14 08:32:02.000000000,,,,,,,,True


In [82]:
dupes_restricted.sort_values(by="clip_filename").head(50)
dupes_restricted["labeling_effort"].value_counts()

labeling_effort
abs_v2_emmanuel             723
evaluation_v2               243
evaluation_v1                46
old_abs_CAC_last_samples     43
old_abs_from_3s_snippets     23
irene_20min                  21
irene_5min                    7
Name: count, dtype: int64

In [83]:
# Drop duplicates keeping the preferred labeling_effort (priority: evaluation_v1 > irene_20min > old_abs_from_3s_snippets > old_abs_CAC_last_samples)
effort_priority = {
    "evaluation_v1": 0,
    "evaluation_v2": 1,  # Treat evaluation_v2 as same priority as evaluation_v1
    "irene_20min":2,
    "irene_5min": 3,

    "old_abs_from_3s_snippets": 4,
    "old_abs_CAC_last_samples": 5   ,
    "abs_v2_emmanuel": 6
}

# Anything not in the map gets lowest priority
merged_df["effort_priority"] = merged_df["labeling_effort"].map(effort_priority).fillna(99).astype(int)

# Sort by clip_filename and effort_priority, so preferred labeling_effort comes first
merged_df = merged_df.sort_values(["clip_filename", "effort_priority"])

# Drop duplicates, keeping the preferred one
merged_df = merged_df.drop_duplicates(subset="clip_filename", keep="first")

# Remove the helper column
merged_df = merged_df.drop(columns=["effort_priority"])

In [84]:

merged_df[merged_df['clip_filename'] == "BSM_20210818_17440200.wav"]

,clip_filename,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,DETAIL,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,DETAILS,boat_labeling_file,boat_labeling_file_id,annotator,SnippetFilename,start_s,end_s
14443,BSM_20210818_17440200.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,,,2021-08-18 17:44:00.000,,,BSM,,201359382,-172.7,2021-08-18 17:44:02.000,2021-08-18 17:44:03.000,,,,,BSM_20210818_174400.wav,1.0,2.0


In [85]:
import os
import wave

# The directory where the wavs are located
wav_dir = "../../data/Verified_Dataset/clip_wavs"

# Check for duplicate clip_filenames in the merged_df
dup_clip_filenames = merged_df['clip_filename'][merged_df['clip_filename'] != ""].duplicated(keep=False)
if dup_clip_filenames.any():
    print("Duplicate clip_filenames found:")
    print(merged_df.loc[dup_clip_filenames, 'clip_filename'].value_counts())
else:
    print("No duplicate clip_filenames in merged_df.")

missing_files = []
not_1s_files = []

# Make sure to only use each filename once for efficiency
unique_filenames = merged_df['clip_filename'].unique()

for fname in unique_filenames:
    if not fname:
        continue  # Skip empty filenames

    wav_path = os.path.join(wav_dir, fname)
    if not os.path.isfile(wav_path):
        missing_files.append(fname)
        continue

    try:
        with wave.open(wav_path, 'r') as wf:
            n_frames = wf.getnframes()
            framerate = wf.getframerate()
            duration = n_frames / float(framerate)
            # Allow small tolerances for floating point/date precision
            if not (0.99 <= duration <= 1.01):
                not_1s_files.append((fname, duration))
    except wave.Error as e:
        print(f"Error opening wave file '{fname}': {e}")
        not_1s_files.append((fname, 'unreadable'))

print(f"Total clip_filenames checked: {len(unique_filenames)}")

if missing_files:
    print(f"Missing wav files ({len(missing_files)}):")
    print(missing_files)
else:
    print("No missing wav files!")

if not_1s_files:
    print(f"Wavs not ~1 second long ({len(not_1s_files)}):")
    for fname, duration in not_1s_files:
        print(f"{fname}: duration={duration}")
else:
    print("All wav files are 1 second long!")



No duplicate clip_filenames in merged_df.
Total clip_filenames checked: 16836
No missing wav files!
Wavs not ~1 second long (1):
BSM_20220731_17130300.wav: duration=0.864625


In [86]:
# Remove all rows in merged_df whose clip_filename is in not_1s_files list
# not_1s_files is a list of (fname, duration) pairs; we want only the filenames
not_1s_fnames = [fname for fname, _ in not_1s_files]
initial_row_count = len(merged_df)
merged_df = merged_df[~merged_df['clip_filename'].isin(not_1s_fnames)].reset_index(drop=True)
print(f"Removed {initial_row_count - len(merged_df)} rows with wav files not ~1s long.")


Removed 1 rows with wav files not ~1s long.


In [87]:
merged_df.groupby("labeling_effort")["BBPC"].value_counts()

labeling_effort           BBPC
abs_v2_emmanuel           0       1317
evaluation_v1             0       3393
                          1        208
evaluation_v2             0       3533
                          1         88
irene_20min               0        661
                          1        535
irene_5min                1        153
                          0        142
old_abs_CAC_last_samples  0        159
old_abs_from_3s_snippets  0       6646
Name: count, dtype: int64

In [94]:
errors = pd.read_csv("../../data/evaluation_snippets/errors.csv")
errors.head()


,clip_filename,GROUNDTRUTH,Unnamed: 2
0,RDL_20200727_13012408.wav,b,NaN
1,BSM_20170724_13444600.wav,e,NaN
2,CAC_20210718_08544200.wav,e,NaN
3,CAC_20210718_08544300.wav,e,NaN
4,KAM_20200730_13292000.wav,h,NaN


In [95]:
# Create FIXED_GROUNDTRUTH column by matching errors with merged_df
# Create a mapping from clip_filename to GROUNDTRUTH from errors
error_corrections = dict(zip(errors['clip_filename'], errors['GROUNDTRUTH']))

# Create FIXED_GROUNDTRUTH column in merged_df using the mapping
merged_df['FIXED_GROUNDTRUTH'] = merged_df['clip_filename'].map(error_corrections)

# Show how many rows were updated
print(f"Number of rows with FIXED_GROUNDTRUTH values: {merged_df['FIXED_GROUNDTRUTH'].notna().sum()}")
print(f"\nFixed errors for these files:")
display(merged_df[merged_df['FIXED_GROUNDTRUTH'].notna()][['clip_filename', 'ECHO', 'BBPC', 'HFPC', 'Whistle', 'FIXED_GROUNDTRUTH']].head(20))

Number of rows with FIXED_GROUNDTRUTH values: 23

Fixed errors for these files:


,clip_filename,ECHO,BBPC,HFPC,Whistle,FIXED_GROUNDTRUTH
413,BSM_20170724_13410900.wav,0,0,0,0,e
414,BSM_20170724_13411000.wav,0,0,0,0,e
444,BSM_20170724_13414000.wav,0,0,0,0,e
467,BSM_20170724_13420300.wav,0,0,0,0,e
478,BSM_20170724_13421400.wav,0,0,0,0,e
487,BSM_20170724_13422300.wav,0,0,0,0,e
488,BSM_20170724_13422400.wav,0,0,0,0,e
499,BSM_20170724_13423500.wav,0,0,0,0,e
522,BSM_20170724_13425800.wav,0,0,0,0,e
538,BSM_20170724_13431400.wav,0,0,0,0,e


In [ ]:
# Apply fix_calls logic only to rows where FIXED_GROUNDTRUTH is not NaN
def fix_calls(gt):
    if pd.isna(gt):
        return pd.Series([False, False, False, False])
    gt_str = str(gt).strip()
    if 'a' in gt_str:
        return pd.Series([False, False, False, False])
    return pd.Series([
        'e' in gt_str,  # ECHO
        'b' in gt_str,  # BBPC
        'h' in gt_str,  # HFPC
        'w' in gt_str   # Whistle
    ])

# Only fix rows where FIXED_GROUNDTRUTH is not NaN
rows_to_fix = merged_df['FIXED_GROUNDTRUTH'].notna()

print(f"Fixing {rows_to_fix.sum()} rows based on FIXED_GROUNDTRUTH values...")

# Apply fix_calls to FIXED_GROUNDTRUTH for those rows
if rows_to_fix.any():
    fixed_values = merged_df.loc[rows_to_fix, 'FIXED_GROUNDTRUTH'].apply(fix_calls)
    fixed_values.columns = ['ECHO', 'BBPC', 'HFPC', 'Whistle']
   
    # Update only those rows in merged_df
    merged_df.loc[rows_to_fix, ['ECHO', 'BBPC', 'HFPC', 'Whistle']] = fixed_values.values
    
    print("\nAfter fixing:")
    display(merged_df.loc[rows_to_fix, ['clip_filename', 'ECHO', 'BBPC', 'HFPC', 'Whistle', 'FIXED_GROUNDTRUTH']].head(10))
    
    # Update Call column as well
    merged_df["Call"] = merged_df[["ECHO", "BBPC", "HFPC", "Whistle"]].any(axis=1).astype(int)
    print("\nCall column updated based on fixed values.")

Fixing 23 rows based on FIXED_GROUNDTRUTH values...

Before fixing:


,clip_filename,ECHO,BBPC,HFPC,Whistle,FIXED_GROUNDTRUTH
413,BSM_20170724_13410900.wav,0,0,0,0,e
414,BSM_20170724_13411000.wav,0,0,0,0,e
444,BSM_20170724_13414000.wav,0,0,0,0,e
467,BSM_20170724_13420300.wav,0,0,0,0,e
478,BSM_20170724_13421400.wav,0,0,0,0,e
487,BSM_20170724_13422300.wav,0,0,0,0,e
488,BSM_20170724_13422400.wav,0,0,0,0,e
499,BSM_20170724_13423500.wav,0,0,0,0,e
522,BSM_20170724_13425800.wav,0,0,0,0,e
538,BSM_20170724_13431400.wav,0,0,0,0,e


TypeError: Invalid value '[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True False  True  True False]' for dtype 'int64'

In [ ]:
def fix_calls(gt):
    
    if pd.isna(gt):
        return pd.Series([False, False, False, False])
    gt_str = str(gt)
    if 'a' in gt_str:
        return pd.Series([False, False, False, False])
    return pd.Series([
        'e' in gt_str,  # ECHO_verif
        'b' in gt_str,  # BBPC_verif
        'h' in gt_str,  # HFPC_verif
        'w' in gt_str   # Whislte_verif
    ])

labels_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = labels_df["GROUNDTRUTH"].apply(set_verif_flags)
# Convert ECHO, BBPC, HFPC, Whistle columns to 0/1 integers
labels_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = labels_df[["ECHO", "BBPC", "HFPC", "Whistle"]].astype(int)

In [89]:
merged_df

,clip_filename,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,DETAIL,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,DETAILS,boat_labeling_file,boat_labeling_file_id,annotator,SnippetFilename,start_s,end_s
0,BSM_20170724_09471700.wav,0,0,0,0,0,1.0,evaluation_v2,,0.0,1.0,,,201359382.170724093002.wav,201359382.170724094717.snippet.wav,2017-07-24 09:47:17,1035.0,1155.0,BSM,../../data/evaluation_snippets/v2/BSM_2017/Boat,201359382,-172.7,2017-07-24 09:47:17,2017-07-24 09:47:18,,201359382.170724093002.Table.1.selections.txt,1.0,VA,,,
1,BSM_20170724_09471800.wav,0,0,0,0,0,1.0,evaluation_v2,,1.0,2.0,,,201359382.170724093002.wav,201359382.170724094717.snippet.wav,2017-07-24 09:47:17,1035.0,1155.0,BSM,../../data/evaluation_snippets/v2/BSM_2017/Boat,201359382,-172.7,2017-07-24 09:47:18,2017-07-24 09:47:19,,201359382.170724093002.Table.1.selections.txt,1.0,VA,,,
2,BSM_20170724_09471900.wav,0,0,0,0,0,1.0,evaluation_v2,,2.0,3.0,,,201359382.170724093002.wav,201359382.170724094717.snippet.wav,2017-07-24 09:47:17,1035.0,1155.0,BSM,../../data/evaluation_snippets/v2/BSM_2017/Boat,201359382,-172.7,2017-07-24 09:47:19,2017-07-24 09:47:20,,201359382.170724093002.Table.1.selections.txt,1.0,VA,,,
3,BSM_20170724_09472000.wav,0,0,0,0,0,1.0,evaluation_v2,,3.0,4.0,,,201359382.170724093002.wav,201359382.170724094717.snippet.wav,2017-07-24 09:47:17,1035.0,1155.0,BSM,../../data/evaluation_snippets/v2/BSM_2017/Boat,201359382,-172.7,2017-07-24 09:47:20,2017-07-24 09:47:21,,201359382.170724093002.Table.1.selections.txt,1.0,VA,,,
4,BSM_20170724_09472100.wav,0,0,0,0,0,1.0,evaluation_v2,,4.0,5.0,,,201359382.170724093002.wav,201359382.170724094717.snippet.wav,2017-07-24 09:47:17,1035.0,1155.0,BSM,../../data/evaluation_snippets/v2/BSM_2017/Boat,201359382,-172.7,2017-07-24 09:47:21,2017-07-24 09:47:22,,201359382.170724093002.Table.1.selections.txt,1.0,VA,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16830,RDL_20200906_14555938.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,5675.200906144958.wav,,2020-09-06 14:55:57.380000,,,RDL,,5675,-176.5,2020-09-06 14:55:59.380000,2020-09-06 14:56:00.380000,,,,,RDL_20200906_14555738.wav,1.0,2.0
16831,RDL_20200906_14560038.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,5675.200906144958.wav,,2020-09-06 14:55:57.380000,,,RDL,,5675,-176.5,2020-09-06 14:56:00.380000,2020-09-06 14:56:01.380000,,,,,RDL_20200906_14555738.wav,2.0,3.0
16832,RDL_20200906_14573769.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,5675.200906144958.wav,,2020-09-06 14:57:36.690000,,,RDL,,5675,-176.5,2020-09-06 14:57:37.690000,2020-09-06 14:57:38.690000,,,,,RDL_20200906_14573669.wav,0.0,1.0
16833,RDL_20200906_14573869.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,5675.200906144958.wav,,2020-09-06 14:57:36.690000,,,RDL,,5675,-176.5,2020-09-06 14:57:38.690000,2020-09-06 14:57:39.690000,,,,,RDL_20200906_14573669.wav,1.0,2.0


In [18]:
merged_df.to_csv("../../data/Verified_Dataset/labels/labels_merged.csv", index=False)